In [1]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=".*DataFrameGroupBy.apply operated on the grouping columns.*"
)

# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [2]:
# Write your code below.

%load_ext dotenv
%dotenv

In [3]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [4]:
import os
from glob import glob

# Write your code below.

PRICE_DATA = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True)
parquet_files


['../../05_src/data/prices/FLIC/FLIC_1995/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_1995/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2014/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2014/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_1993/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_1993/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2020/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2020/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2018/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2018/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2011/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2011/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2001/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2001/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2008/part.0.parquet',
 '../../05_src/data/prices/FLIC/FLIC_2008/part.1.parquet',
 '../../05_src/data/prices/FLIC/FLIC_1988/part.0.parquet

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [5]:
# Write your code below.

dd_prices = dd.read_parquet(parquet_files)

if "Adj Close" in dd_prices.columns:
    dd_prices = dd_prices.rename(columns={'Adj Close':'Adj_Close'})

def add_features(pdf):
    pdf = pdf.sort_values(['Date']).copy()
    
    pdf['Close_lag_1'] = pdf['Close'].shift(1)
    pdf['Adj_Close_lag_1'] = pdf['Adj_Close'].shift(1)
    pdf['returns'] = (pdf['Close'] / pdf['Close_lag_1']) - 1
    pdf['hi_lo_range'] = pdf['High'] - pdf['Low']

    return pdf

meta = add_features(dd_prices._meta.copy())

dd_feat = dd_prices.groupby('ticker', group_keys=False).apply(add_features, meta=meta)

In [6]:
meta.dtypes

Date                datetime64[ns]
Open                       float64
High                       float64
Low                        float64
Close                      float64
Adj_Close                  float64
Volume                     float64
source             string[pyarrow]
ticker             string[pyarrow]
Year                         int32
Close_lag_1                float64
Adj_Close_lag_1            float64
returns                    float64
hi_lo_range                float64
dtype: object

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [7]:
# Write your code below.

df_dd_pandas = dd_feat.compute() # converts to pandas
df_dd_pandas = df_dd_pandas.sort_values(['ticker', 'Date'])
df_dd_pandas['moving_average_10_days'] = (
    df_dd_pandas
    .groupby('ticker')['returns']
    .transform(lambda x:x.rolling(10).mean())
)
df_dd_pandas

,Date,Open,High,Low,Close,Adj_Close,Volume,source,ticker,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,moving_average_10_days
219014,2001-04-06,17.000000,17.000000,16.781250,16.828125,8.672449,162100.0,ABB.csv,ABB,2001,NaN,NaN,NaN,0.218750,NaN
219015,2001-04-09,16.900000,17.299999,16.900000,17.299999,8.915632,31300.0,ABB.csv,ABB,2001,16.828125,8.672449,0.028041,0.400000,NaN
219016,2001-04-10,17.750000,17.920000,17.700001,17.790001,9.168156,39500.0,ABB.csv,ABB,2001,17.299999,8.915632,0.028324,0.219999,NaN
219017,2001-04-11,17.500000,17.600000,17.400000,17.600000,9.070239,17900.0,ABB.csv,ABB,2001,17.790001,9.168156,-0.010680,0.200001,NaN
219018,2001-04-12,17.500000,17.500000,17.400000,17.500000,9.018702,33000.0,ABB.csv,ABB,2001,17.600000,9.070239,-0.005682,0.100000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2224,2020-03-26,33.070000,34.880001,32.950001,34.820000,34.820000,2670700.0,YNDX.csv,YNDX,2020,33.419998,33.419998,0.041891,1.930000,0.009967
2225,2020-03-27,32.889999,33.779999,32.389999,32.980000,32.980000,2413100.0,YNDX.csv,YNDX,2020,34.820000,34.820000,-0.052843,1.389999,-0.000984
2226,2020-03-30,33.310001,33.930000,32.779999,33.880001,33.880001,2973200.0,YNDX.csv,YNDX,2020,32.980000,32.980000,0.027289,1.150002,0.010997
2227,2020-03-31,34.000000,35.220001,33.759998,34.049999,34.049999,3245700.0,YNDX.csv,YNDX,2020,33.880001,33.880001,0.005018,1.460003,0.008706


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No, it was not strictly necessary to convert to pandas to calculate the moving average return. 
Dask can calculate rolling features, but it requires care because the data is partitioned. 
If the same ticker is split across partitions, rolling windows and lags can be incorrect unless the data is properly grouped, sorted, and partitioned.

Using pandas is simpler when the computed data fits in memory. 
Using Dask is better when the dataset is too large for memory or when we want parallel/distributed computation.

In [8]:
# Optional: Dask-based version for comparison, not required for the assignment.

def add_moving_average(pdf):
    pdf = pdf.sort_values('Date').copy()
    pdf['moving_average_10_days'] = pdf['returns'].rolling(10).mean()
    return pdf

meta = dd_feat._meta.copy()
meta['moving_average_10_days'] = meta['returns'].astype('float64')

dd_feat_ma = (
    dd_feat
    .groupby('ticker', group_keys=False)
    .apply(add_moving_average, meta=meta)
)

df_dd = dd_feat_ma.compute() # Let's convert to pandas just to see the results
df_dd

,Date,Open,High,Low,Close,Adj_Close,Volume,source,ticker,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,moving_average_10_days
138282,2006-10-25,20.000000,20.010000,20.000000,20.010000,4.756574,381300.0,IRR.csv,IRR,2006,NaN,NaN,NaN,0.010000,NaN
138283,2006-10-26,20.010000,20.240000,20.010000,20.150000,4.789854,52800.0,IRR.csv,IRR,2006,20.010000,4.756574,0.006996,0.230000,NaN
138284,2006-10-27,20.150000,20.260000,20.010000,20.020000,4.758952,32000.0,IRR.csv,IRR,2006,20.150000,4.789854,-0.006452,0.250000,NaN
138285,2006-10-30,20.010000,20.150000,20.010000,20.100000,4.777967,45500.0,IRR.csv,IRR,2006,20.020000,4.758952,0.003996,0.139999,NaN
138286,2006-10-31,20.139999,20.139999,20.049999,20.080000,4.773212,23800.0,IRR.csv,IRR,2006,20.100000,4.777967,-0.000995,0.090000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2224,2020-03-26,33.070000,34.880001,32.950001,34.820000,34.820000,2670700.0,YNDX.csv,YNDX,2020,33.419998,33.419998,0.041891,1.930000,0.009967
2225,2020-03-27,32.889999,33.779999,32.389999,32.980000,32.980000,2413100.0,YNDX.csv,YNDX,2020,34.820000,34.820000,-0.052843,1.389999,-0.000984
2226,2020-03-30,33.310001,33.930000,32.779999,33.880001,33.880001,2973200.0,YNDX.csv,YNDX,2020,32.980000,32.980000,0.027289,1.150002,0.010997
2227,2020-03-31,34.000000,35.220001,33.759998,34.049999,34.049999,3245700.0,YNDX.csv,YNDX,2020,33.880001,33.880001,0.005018,1.460003,0.008706


## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [x] Created a branch with the correct naming convention.
- [x] Ensured that the repository is public.
- [x] Reviewed the PR description guidelines and adhered to them.
- [x] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.